## Приоритезация обращений. Model

In [86]:
# подхватывать правки локальных модулей без рестарта ядра
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import optuna
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# локальные модули
import preprocessor

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [87]:
ROOT = Path(".")
DATA_DIR = ROOT / "data"

TARGET = "target"
RANDOM_STATE = 42

TEST_DAYS = 3  # test
VALID_DAYS = 2  # val
N_FOLDS = 3  # число фолдов
MIN_TRAIN_DAYS = 5  # минимальный размер train без val

SEEDS = (0, 1, 2)  # для усреднения по сидам
TUNE_SEEDS = (0, 1)

### Загрузка данных

Загружаем обучающую выборку, тестовую выборку и события.

In [88]:
train = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
events = pd.read_csv(DATA_DIR / "events.csv")

print("train:", train.shape)
print("test_df:", test_df.shape)
print("events:", events.shape)

train: (13694, 119)
test_df: (4306, 118)
events: (254705, 7)


### Выбор признаков TODO:

### Валидация

`test_df` лежит строго после `train` по времени:
- `train 07.04-22.04 = 16 дней`
- `test_df 23.04-27.04 = 5 дней`

#### Метрики

In [89]:
def daily_ap(y_true, scores, dates) -> float:
    """Метрика соревнования: AP считается внутри каждого дня и усредняется по дням.

    Дни без единого положительного примера пропускаем - AP на них не определён.
    """
    df = pd.DataFrame({"y": y_true, "s": scores, "d": dates})
    per_day = [average_precision_score(g["y"], g["s"])
               for _, g in df.groupby("d") if g["y"].sum() > 0]
    return float(np.mean(per_day))

#### Схема

Обычная кросс-валидация не подходит. Двигаемся скользящим окном по дням, чтобы не было data leak. **Минимальный квант --- 1 день**, т.к. метрика Daily AP.

Фолд --- это **тройка подряд идущих блоков** `train | valid | test`, которая едет от конца выборки к началу. `valid` стоит вплотную перед `test`: на нём подбираются гиперпараметры этого же фолда, и подбор всегда смотрит на дни, непосредственно предшествующие замеру, --- ровно так же, как перед сабмитом.

```
fold 0: train 07.04-17.04 (11д) | valid 18-19 | test 20-22
fold 1: train 07.04-14.04  (8д) | valid 15-16 | test 17-19
fold 2: train 07.04-11.04  (5д) | valid 12-13 | test 14-16
```

Шаг тройки равен `TEST_DAYS`, поэтому **тест-блоки не перекрываются**: три замера сделаны на разных днях и их можно усреднять. val-блоки тоже не пересекаются с тестами своего фолда => параметры подбираются честно.

**Expanding, а не rolling**: данных мало, роскоши дропать начало выборки нет => абсолютные скоры фолдов между собой несравнимы, поэтому решения принимаем по разностям моделей внутри фолда, а не по разнице средних.

In [91]:
def folds(df, valid_days=VALID_DAYS, test_days=TEST_DAYS, n_folds=N_FOLDS, min_train_days=MIN_TRAIN_DAYS, step_days=None):
    """
    Тройки (train, valid, test) из подряд идущих дней, едущие к началу выборки.
    
    - step_days=None ---> шаг равен длине тест-блока, то есть тесты не перекрываются.
    """
    step_days = step_days or test_days
    days = pd.to_datetime(df["assignment_date"]).dt.date
    ordered = sorted(days.unique())

    for k in range(n_folds):
        test_end = len(ordered) - k * step_days
        test_start = test_end - test_days
        valid_start = test_start - valid_days

        if valid_start < min_train_days:
            raise ValueError(f"фолд {k} не помещается: на train осталось бы {valid_start}д "
                             f"при минимуме {min_train_days}")

        yield (df[days < ordered[valid_start]],
               df[days.isin(ordered[valid_start:test_start])],
               df[days.isin(ordered[test_start:test_end])])


show = lambda df: sorted(pd.to_datetime(df.assignment_date).dt.date.unique())

for i, (tr, va, te) in enumerate(folds(train)):
    total = len(tr) + len(va) + len(te)
    part = lambda df, label: (f"{label} {show(df)[0]}..{show(df)[-1]} "
                              f"({len(show(df))}д, {len(df)} строк, {len(df) / total:.0%})")
    print(f"fold {i}: {part(tr, 'train')} | {part(va, 'valid')} | {part(te, 'test')}")

fold 0: train 2026-04-07..2026-04-17 (11д, 9397 строк, 69%) | valid 2026-04-18..2026-04-19 (2д, 1738 строк, 13%) | test 2026-04-20..2026-04-22 (3д, 2559 строк, 19%)
fold 1: train 2026-04-07..2026-04-14 (8д, 6743 строк, 61%) | valid 2026-04-15..2026-04-16 (2д, 1753 строк, 16%) | test 2026-04-17..2026-04-19 (3д, 2639 строк, 24%)
fold 2: train 2026-04-07..2026-04-11 (5д, 4255 строк, 50%) | valid 2026-04-12..2026-04-13 (2д, 1646 строк, 19%) | test 2026-04-14..2026-04-16 (3д, 2595 строк, 31%)


#### Использование

Решения принимаем по **парным дельтам** --- разности моделей внутри одного фолда.

In [92]:
def fit_score(build_model, fit_df, eval_df, seeds=SEEDS) -> list:
    """Учим на fit_df, меряем daily AP на eval_df. Одно значение на сид.

    В модель передаём строки целиком, а не срез по признакам: препроцессору нужны
    lead_id и assignment_ts, чтобы приджойнить события до момента назначения.
    """
    scores = []
    for seed in seeds:
        model = build_model(seed).fit(fit_df, fit_df[TARGET])
        proba = model.predict_proba(eval_df)[:, 1]
        scores.append(daily_ap(eval_df[TARGET], proba, eval_df["assignment_date"]))
    return scores


### Модель

#### Модель (#0.Baseline LogReg + #1. LogReg)

Используем простую Logistic Regression:

- числовые признаки: заполнение пропусков медианой и scaling;
- категориальные признаки: заполнение самым частым значением и one-hot encoding;
- `class_weight="balanced"` из-за дисбаланса классов.

`baseline` без изменений из `quickstart.ipynb`, `logreg` имеет препроцессинг признаков 

In [97]:
def build_baseline(seed=RANDOM_STATE, **params):
    """Логрег детерминирован, сид на него не влияет - но интерфейс общий с бустингом."""
    return preprocessor.Pipeline(
        events=events,
        baseline=True,
        model=LogisticRegression(random_state=seed, max_iter=1000,
                                 class_weight="balanced", **params),
    )

In [93]:
def build_logreg(seed=RANDOM_STATE, **params):
    """Логрег детерминирован, сид на него не влияет - но интерфейс общий с бустингом."""
    return preprocessor.Pipeline(
        events=events,
        model=LogisticRegression(random_state=seed, max_iter=1000,
                                 class_weight="balanced", **params),
    )

#### Модель (#2. Catboost)

Логрегрессии нужен полный препроцессинг, бустингу он наоборот вредит. Поэтому CatBoost собираем с `sklearn_preprocess=False`, то есть без `ColumnTransformer`:

- **пропуски не импутирую** --- CatBoost обрабатывает `NaN` нативно, а замена медианой стёрла бы саму информацию о пропуске;
- **не делаю One-Hot** --- вместо него встроенная обработка категориальных признаков;
- **`boosting_type="Plain"`** вместо `Ordered`, который CatBoost включает сам на малых выборках --- быстрее и не хуже

In [94]:
def build_catboost(seed=RANDOM_STATE, **params):
    """CatBoost без sklearn-препроцессинга. Пустой params => дефолтная конфигурация."""
    return preprocessor.Pipeline(
        events=events,
        sklearn_preprocess=False,
        model=CatBoostClassifier(
            random_seed=seed,
            auto_class_weights="Balanced",
            cat_features=preprocessor.CATEGORICAL_COLUMNS,  # NaN в категориальных нет
            boosting_type="Plain",
            verbose=False,
            **params,
        ),
    )

#### Модель (#3. Бэггинг Catboost + LightGBM/XGBoost)

**Идея**: данные фреймворки по разному строят деревья при реализации GB => их аггрегация может дать лучший результат

TODO:

#### Подбор гиперпараметров (optuna + Catboost)

Подбор только для `CatBoost`. 

TPE, а не сетка: пространство непрерывное.

`tune` принимает `train` и `valid` **одного фолда**, поэтому у каждого фолда набор параметров свой + усредняем значение по сидам.

In [95]:
N_TRIALS = 20
optuna.logging.set_verbosity(optuna.logging.WARNING)

def suggest_params(trial) -> dict:
    return {
        "iterations": trial.suggest_int("iterations", 200, 800, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.5, 5.0),
    }

def tune(train_part, valid_part, n_trials=N_TRIALS, seeds=TUNE_SEEDS):
    """optuna на valid-блоке одного фолда. Возвращает study целиком: best_params
    идут в замер, а разброс лучших конфигураций между фолдами - в диагностику."""
    def objective(trial):
        params = suggest_params(trial)
        build = lambda seed: build_catboost(seed, **params)
        return float(np.mean(fit_score(build, train_part, valid_part, seeds=seeds)))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(objective, n_trials=n_trials)
    return study

#### Сравнение результатов моделей

Один проход по фолдам: на каждом подбираем параметры для кэтбуста на его `valid`, дообучаемся на `train + valid` и меряем все модели на его `test`.

Ориентир из бейзлайна (`make_validation_split` в `quickstart.ipynb`, обычный split 80/20 по датам): AP **0.48506**. Напрямую не сравнить - другая схема валидации, но совпадает `baseline` модель с `quickstart.ipynb`.

In [101]:
def walk_forward(df, n_trials=N_TRIALS):
    """Проход по фолдам: свой подбор на valid каждого фолда, замер на его test."""
    rows, chosen = [], {}

    for k, (tr, va, te) in enumerate(folds(df)):
        study = tune(tr, va, n_trials=n_trials)
        chosen[f"fold {k}"] = study.best_params

        # дообучаемся на valid перед замером: тот же режим, что и перед сабмитом
        fit_df = pd.concat([tr, va])
        builders = {
            "baseline": build_baseline,
            "logreg": build_logreg,
            "catboost_default": build_catboost,
            "catboost_tuned": lambda seed: build_catboost(seed, **study.best_params),
        }
        for name, build in builders.items():
            for score in fit_score(build, fit_df, te):
                rows.append({"fold": k, "model": name, "daily_ap": score})

        print(f"fold {k}: test {show(te)[0]}..{show(te)[-1]}  "
              f"на valid best={study.best_value:.4f}")

    return pd.DataFrame(rows), pd.DataFrame(chosen).T


In [ ]:
results, best_params = walk_forward(train)

per_fold = results.groupby(["fold", "model"])["daily_ap"].mean().unstack()
display(per_fold.round(4))
display(per_fold.agg(["mean", "std"]).round(4))

fold 0: test 2026-04-20..2026-04-22  на valid best=0.6307
fold 1: test 2026-04-17..2026-04-19  на valid best=0.5643
fold 2: test 2026-04-14..2026-04-16  на valid best=0.5979


model,catboost_default,catboost_tuned,logreg
fold,,,
0,0.6182,0.6263,0.5471
1,0.6168,0.6167,0.5443
2,0.5636,0.5656,0.4906


model,catboost_default,catboost_tuned,logreg
mean,0.5995,0.6029,0.5274
std,0.0312,0.0327,0.0319


In [98]:
delta = per_fold["catboost_tuned"] - per_fold["catboost_default"]
noise = results.groupby(["fold", "model"])["daily_ap"].std().max()

print(f"tuned - default по фолдам: {', '.join(f'{d:+.4f}' for d in delta)}")
print(f"среднее {delta.mean():+.4f}   разброс по сидам внутри фолда, макс sd = {noise:.4f}")

tuned - default по фолдам: +0.0081, -0.0002, +0.0020
среднее +0.0033   разброс по сидам внутри фолда, макс sd = 0.0050


### Submission

Обучаем модель на всем train и строим score для test_df.
Файл для отправки должен содержать две колонки: `lead_id` и `score`.

In [103]:
FINAL_MODEL = "catboost_tuned"
final_params = best_params.loc["fold 0"].to_dict()
builders = {
    "baseline": build_baseline,
    "logreg": build_logreg,
    "catboost_default": build_catboost,
    "catboost_tuned": lambda seed: build_catboost(seed, **final_params),
}
final_params

{'iterations': 800.0,
 'learning_rate': 0.09076241634923791,
 'depth': 5.0,
 'l2_leaf_reg': 9.12621062005872,
 'random_strength': 3.4497986920450834}

In [104]:
final_model = builders[FINAL_MODEL](RANDOM_STATE).fit(train, train[TARGET])

In [105]:
submission = pd.DataFrame({
    "lead_id": test_df["lead_id"].astype(str),
    "score": final_model.predict_proba(test_df)[:, 1],
})
submission.to_csv(ROOT / "submission.csv", index=False)
submission.head()

,lead_id,score
0,lead_97e409eb8f8c8246,0.048937
1,lead_55310edb4489f9e9,0.422287
2,lead_e7f653a2c6a7eee8,0.760697
3,lead_22f8e1cfc487ac20,0.074602
4,lead_48b638b839abfac3,0.151864


In [106]:
# Минимальные проверки формата перед загрузкой.
assert list(submission.columns) == ["lead_id", "score"]
assert len(submission) == len(test_df)
assert submission["lead_id"].is_unique
assert submission["score"].between(0, 1).all()

print("submission.csv is ready")

submission.csv is ready
